## ⚙️ 0 — Install Dependencies

In [2]:
!pip install lightgbm xgboost catboost optuna shap scikit-learn pandas numpy joblib --quiet



## ⚙️ 1 — Imports 



In [3]:
import os, sys, json, time, warnings

import numpy as np
import pandas as pd
import joblib
from pathlib import Path
from datetime import datetime
from dataclasses import dataclass, field
from typing import List, Dict, Any, Optional, Tuple

import optuna
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier  # ADD THIS
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings("ignore")

if str(Path(".").resolve()) not in sys.path:
    sys.path.insert(0, str(Path(".").resolve()))


c:\Users\HP\anaconda3\envs\ai_project\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Step 2: Configuration (Update)


In [4]:
@dataclass
class PipelineConfig:
    base_dir  : Path = Path(".")
    data_dir  : Path = Path("data/raw")
    model_dir : Path = Path("models")

    use_optuna    : bool = True
    optuna_trials : int  = 30  
    cv_folds      : int  = 5
    random_state  : int  = 42
    shap_sample   : int  = 1000

    lgb_defaults : Dict = field(default_factory=lambda: {
        "objective": "binary", "metric": "auc", "verbosity": -1,
        "n_estimators": 1500, "learning_rate": 0.02, "num_leaves": 63,
        "min_child_samples": 50, "subsample": 0.8, "colsample_bytree": 0.8,
        "reg_alpha": 0.1, "reg_lambda": 1.0, "scale_pos_weight": 3.0,
        "n_jobs": -1, "random_state": 42,
    })

    xgb_defaults : Dict = field(default_factory=lambda: {
        "objective": "binary:logistic", "eval_metric": "auc",
        "n_estimators": 1500, "learning_rate": 0.02, "max_depth": 6,
        "min_child_weight": 5, "subsample": 0.8, "colsample_bytree": 0.8,
        "gamma": 0.1, "reg_alpha": 0.1, "reg_lambda": 1.0,
        "tree_method": "hist", "n_jobs": -1, "random_state": 42,
    })
    
    # ADD THIS NEW SECTION:
    cb_defaults : Dict = field(default_factory=lambda: {
        "iterations": 1500,
        "learning_rate": 0.02,
        "depth": 6,
        "l2_leaf_reg": 3.0,
        "random_seed": 42,
        "verbose": False,
        "task_type": "CPU",
    })

CFG = PipelineConfig()
CFG.model_dir.mkdir(exist_ok=True)


---
Cell 3 - DataLoader: ADD these aggregations



In [8]:
class DataLoader:
    def __init__(self, config: PipelineConfig):
        self.cfg = config

    def _agg_bureau(self) -> pd.DataFrame:
        bureau = pd.read_csv(self.cfg.data_dir / "bureau.csv")
        bb = pd.read_csv(self.cfg.data_dir / "bureau_balance.csv")
        bb_agg = bb.groupby("SK_ID_BUREAU")["MONTHS_BALANCE"].mean().reset_index()
        bb_agg.columns = ["SK_ID_BUREAU", "BB_MONTHS_BALANCE_mean"]
        bureau = bureau.merge(bb_agg, on="SK_ID_BUREAU", how="left")
        return bureau.groupby("SK_ID_CURR").agg(
            BUREAU_AMT_CREDIT_SUM_mean=("AMT_CREDIT_SUM", "mean"),
            BUREAU_AMT_CREDIT_SUM_sum=("AMT_CREDIT_SUM", "sum"),
            BUREAU_AMT_CREDIT_SUM_max=("AMT_CREDIT_SUM", "max"),
            BUREAU_AMT_CREDIT_SUM_DEBT_mean=("AMT_CREDIT_SUM_DEBT", "mean"),
            BUREAU_AMT_CREDIT_SUM_DEBT_sum=("AMT_CREDIT_SUM_DEBT", "sum"),
            BUREAU_CREDIT_DAY_OVERDUE_max=("CREDIT_DAY_OVERDUE", "max"),
            BUREAU_DAYS_CREDIT_mean=("DAYS_CREDIT", "mean"),
            BUREAU_DAYS_CREDIT_min=("DAYS_CREDIT", "min"),
            BUREAU_BB_MONTHS_BALANCE_mean_mean=("BB_MONTHS_BALANCE_mean", "mean"),
            BUREAU_CNT=("SK_ID_BUREAU", "count"),  #NEW
        ).reset_index()

    def _agg_previous(self) -> pd.DataFrame:
        prev = pd.read_csv(self.cfg.data_dir / "previous_application.csv")
        return prev.groupby("SK_ID_CURR").agg(
            PREV_AMT_APPLICATION_mean=("AMT_APPLICATION","mean"),
            PREV_AMT_APPLICATION_max=("AMT_APPLICATION","max"),
            PREV_AMT_CREDIT_mean=("AMT_CREDIT", "mean"),
            PREV_AMT_CREDIT_max=("AMT_CREDIT", "max"),
            PREV_AMT_ANNUITY_mean=("AMT_ANNUITY", "mean"),
            PREV_CNT_PAYMENT_mean=("CNT_PAYMENT", "mean"),
            PREV_AMT_DOWN_PAYMENT_mean=("AMT_DOWN_PAYMENT", "mean"),  #NEW
            PREV_DAYS_DECISION_mean=("DAYS_DECISION", "mean"),  #NEW
            PREV_CNT=("SK_ID_PREV", "count"),  #NEW
        ).reset_index()

    def _agg_installments(self) -> pd.DataFrame:
        inst = pd.read_csv(self.cfg.data_dir / "installments_payments.csv")
        inst["PAYMENT_DIFF"] = inst["AMT_INSTALMENT"] - inst["AMT_PAYMENT"]
        inst["DAYS_DIFF"] = inst["DAYS_ENTRY_PAYMENT"] - inst["DAYS_INSTALMENT"]  #NEW
        inst["PAYMENT_RATIO"] = inst["AMT_PAYMENT"] / (inst["AMT_INSTALMENT"] + 1)  #NEW
        return inst.groupby("SK_ID_CURR").agg(
            PAYMENT_DIFF_mean=("PAYMENT_DIFF", "mean"),
            PAYMENT_DIFF_std=("PAYMENT_DIFF", "std"),  #NEW
            AMT_PAYMENT_mean=("AMT_PAYMENT", "mean"),
            AMT_INSTALMENT_mean=("AMT_INSTALMENT", "mean"),
            DAYS_DIFF_mean=("DAYS_DIFF", "mean"),  #NEW
            PAYMENT_RATIO_mean=("PAYMENT_RATIO", "mean"),  #NEW
        ).reset_index()

    def _agg_pos(self) -> pd.DataFrame:
        pos = pd.read_csv(self.cfg.data_dir / "POS_CASH_balance.csv")
        return pos.groupby("SK_ID_CURR").agg(
            POS_MONTHS_BALANCE_mean=("MONTHS_BALANCE", "mean"),
            POS_MONTHS_BALANCE_min=("MONTHS_BALANCE", "min"),
            POS_CNT_INSTALMENT_mean=("CNT_INSTALMENT", "mean"),
            POS_CNT_INSTALMENT_FUTURE_mean=("CNT_INSTALMENT_FUTURE", "mean"),
            POS_SK_DPD_mean=("SK_DPD", "mean"),  #NEW
            POS_SK_DPD_max=("SK_DPD", "max"),  #NEW
        ).reset_index()

    def _agg_cc(self) -> pd.DataFrame:
        cc = pd.read_csv(self.cfg.data_dir / "credit_card_balance.csv")
        return cc.groupby("SK_ID_CURR").agg(
        CC_AMT_BALANCE_mean=("AMT_BALANCE", "mean"),
        CC_AMT_BALANCE_max=("AMT_BALANCE", "max"),
        CC_AMT_CREDIT_LIMIT_ACTUAL_mean=("AMT_CREDIT_LIMIT_ACTUAL", "mean"),
        CC_AMT_DRAWINGS_CURRENT_mean=("AMT_DRAWINGS_CURRENT", "mean"),
    ).reset_index()


    def run(self) -> pd.DataFrame:
        t0 = time.time()
        df = pd.read_csv(self.cfg.data_dir / "application_train.csv")
        for fn, name in [
            (self._agg_bureau, "bureau"),
            (self._agg_previous, "previous"),
            (self._agg_installments, "installments"),
            (self._agg_pos, "POS"),
            (self._agg_cc, "CC"),
        ]:
            df = df.merge(fn(), on="SK_ID_CURR", how="left")
        print(f"Data loaded: {df.shape[0]} rows, {df.shape[1]} cols")
        return df

loader = DataLoader(CFG)
df_raw = loader.run()


Data loaded: 307511 rows, 157 cols


---
## ⚙️ 4 — FeatureEngineer



In [9]:
class FeatureEngineer:
    def __init__(self, config: PipelineConfig):
        self.cfg = config
        self.new_features: List[str] = []

    def _add(self, df, name, values):
        df[name] = values
        self.new_features.append(name)
        return df

    def run(self, df: pd.DataFrame) -> pd.DataFrame:
        self.new_features = []
        
        # Original features (keep)
        df = self._add(df, "CREDIT_INCOME_RATIO", df["AMT_CREDIT"] / (df["AMT_INCOME_TOTAL"] + 1))
        df = self._add(df, "ANNUITY_INCOME_RATIO", df["AMT_ANNUITY"] / (df["AMT_INCOME_TOTAL"] + 1))
        df = self._add(df, "CREDIT_TERM", df["AMT_ANNUITY"] / (df["AMT_CREDIT"] + 1))
        df = self._add(df, "INCOME_PER_PERSON", df["AMT_INCOME_TOTAL"] / (df["CNT_FAM_MEMBERS"] + 1))
        df = self._add(df, "GOODS_CREDIT_RATIO", df["AMT_GOODS_PRICE"] / (df["AMT_CREDIT"] + 1))
        
        # Original EXT_SOURCE (keep)
        ext = df[["EXT_SOURCE_1","EXT_SOURCE_2","EXT_SOURCE_3"]]
        df = self._add(df, "EXT_SOURCE_MEAN", ext.mean(axis=1))
        df = self._add(df, "EXT_SOURCE_STD", ext.std(axis=1))
        df = self._add(df, "EXT_SOURCE_PROD", df["EXT_SOURCE_1"] * df["EXT_SOURCE_2"] * df["EXT_SOURCE_3"])
        df = self._add(df, "EXT_SOURCE_MIN", ext.min(axis=1))
        df = self._add(df, "EXT_SOURCE_MAX", ext.max(axis=1))
        
        # NEW: Polynomial EXT_SOURCE
        df = self._add(df, "EXT_SOURCE_1_SQ", df["EXT_SOURCE_1"] ** 2)
        df = self._add(df, "EXT_SOURCE_2_SQ", df["EXT_SOURCE_2"] ** 2)
        df = self._add(df, "EXT_SOURCE_3_SQ", df["EXT_SOURCE_3"] ** 2)
        df = self._add(df, "EXT_SOURCE_1_2_PROD", df["EXT_SOURCE_1"] * df["EXT_SOURCE_2"])
        df = self._add(df, "EXT_SOURCE_2_3_PROD", df["EXT_SOURCE_2"] * df["EXT_SOURCE_3"])
        df = self._add(df, "EXT_SOURCE_1_3_PROD", df["EXT_SOURCE_1"] * df["EXT_SOURCE_3"])
        
        # Original age/employment (keep)
        df = self._add(df, "DAYS_BIRTH_YEARS", np.abs(df["DAYS_BIRTH"]) / 365)
        df = self._add(df, "DAYS_EMPLOYED_YEARS", np.abs(df["DAYS_EMPLOYED"].clip(upper=0)) / 365)
        df = self._add(df, "DAYS_EMPLOYED_RATIO", df["DAYS_EMPLOYED"] / (df["DAYS_BIRTH"] - 1))
        df = self._add(df, "EMPLOYED_TO_AGE", df["DAYS_EMPLOYED_YEARS"] / (df["DAYS_BIRTH_YEARS"] + 1))
        
        # NEW: Polynomial age/employment
        df = self._add(df, "AGE_SQ", df["DAYS_BIRTH_YEARS"] ** 2)
        df = self._add(df, "EMPLOYED_SQ", df["DAYS_EMPLOYED_YEARS"] ** 2)
        
        # Original doc/contact (keep)
        doc_cols = [c for c in df.columns if c.startswith("FLAG_DOCUMENT_")]
        contact_cols = ["FLAG_MOBIL","FLAG_EMP_PHONE","FLAG_WORK_PHONE","FLAG_PHONE","FLAG_EMAIL"]
        contact_cols = [c for c in contact_cols if c in df.columns]
        df = self._add(df, "DOCUMENT_COUNT", df[doc_cols].sum(axis=1))
        df = self._add(df, "CONTACT_COUNT", df[contact_cols].sum(axis=1))
        
        # Original bureau (keep)
        df = self._add(df, "BUREAU_DEBT_RATIO",
            df.get("BUREAU_AMT_CREDIT_SUM_DEBT_sum", pd.Series(0, index=df.index)) /
            (df.get("BUREAU_AMT_CREDIT_SUM_sum", pd.Series(1, index=df.index)) + 1))
        
        # NEW: Additional features
        df = self._add(df, "BUREAU_CREDIT_PER_LOAN",
            df.get("BUREAU_AMT_CREDIT_SUM_sum", pd.Series(0, index=df.index)) /
            (df.get("BUREAU_CNT", pd.Series(1, index=df.index)) + 1))
        df = self._add(df, "PREV_APP_TO_CREDIT_RATIO",
            df.get("PREV_AMT_APPLICATION_mean", pd.Series(0, index=df.index)) / (df["AMT_CREDIT"] + 1))
        df = self._add(df, "PREV_CREDIT_DIFF",
            df.get("PREV_AMT_CREDIT_mean", pd.Series(0, index=df.index)) - df["AMT_CREDIT"])
        df = self._add(df, "INSTALLMENT_TO_ANNUITY",
            df.get("AMT_INSTALMENT_mean", pd.Series(0, index=df.index)) / (df["AMT_ANNUITY"] + 1))
        df = self._add(df, "PAYMENT_TO_ANNUITY",
            df.get("AMT_PAYMENT_mean", pd.Series(0, index=df.index)) / (df["AMT_ANNUITY"] + 1))
        df = self._add(df, "POS_DPD_RATIO",
            df.get("POS_SK_DPD_mean", pd.Series(0, index=df.index)) / 
            (df.get("POS_SK_DPD_max", pd.Series(1, index=df.index)) + 1))
        df = self._add(df, "CC_UTILIZATION",
            df.get("CC_AMT_BALANCE_mean", pd.Series(0, index=df.index)) /
            (df.get("CC_AMT_CREDIT_LIMIT_ACTUAL_mean", pd.Series(1, index=df.index)) + 1))
        
        # Binary encode
        df["FLAG_OWN_CAR"] = (df["FLAG_OWN_CAR"] == "Y").astype(int)
        df["FLAG_OWN_REALTY"] = (df["FLAG_OWN_REALTY"] == "Y").astype(int)
        
        # NEW: Binary flags
        df = self._add(df, "HAS_EXT_SOURCE_1", (df["EXT_SOURCE_1"].notna() & (df["EXT_SOURCE_1"] > 0)).astype(int))
        df = self._add(df, "HAS_EXT_SOURCE_2", (df["EXT_SOURCE_2"].notna() & (df["EXT_SOURCE_2"] > 0)).astype(int))
        df = self._add(df, "HAS_EXT_SOURCE_3", (df["EXT_SOURCE_3"].notna() & (df["EXT_SOURCE_3"] > 0)).astype(int))

        print(f"Added {len(self.new_features)} features")
        return df

engineer = FeatureEngineer(CFG)
df_feat = engineer.run(df_raw)


Added 35 features


---
## 5 — Preprocessor


In [10]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder
from utils.named_pipeline import NamedPipeline

class Preprocessor:
    TARGET = "TARGET"
    DROP = ["SK_ID_CURR", "TARGET"]

    def __init__(self, config: PipelineConfig):
        self.cfg = config
        self.le_dict: Dict[str, LabelEncoder] = {}
        self.pipeline: Optional[NamedPipeline] = None
        self.feat_cols: List[str] = []

    def run(self, df: pd.DataFrame) -> Tuple[np.ndarray, np.ndarray, List[str], NamedPipeline]:
        cat_cols = df.select_dtypes("object").columns.tolist()
        num_cols = [c for c in df.columns if c not in self.DROP and c not in cat_cols]

        for col in cat_cols:
            le = LabelEncoder()
            df[col] = le.fit_transform(df[col].fillna("MISSING").astype(str))
            self.le_dict[col] = le

        self.feat_cols = num_cols + cat_cols
        X_df = df[self.feat_cols]
        y = df[self.TARGET].values

        self.pipeline = NamedPipeline([("imputer", SimpleImputer(strategy="median"))])
        self.pipeline.fit(X_df)
        X = self.pipeline.transform(X_df)

        print(f"Features: {X.shape[1]}, Samples: {X.shape[0]}")
        return X, y, self.feat_cols, self.pipeline

preprocessor_step = Preprocessor(CFG)
X, y, feat_cols, pipeline = preprocessor_step.run(df_feat)


Features: 190, Samples: 307511


Step 6: Hyperparameter Tuner (Add CatBoost)

In [11]:
class HyperparamTuner:
    def __init__(self, config: PipelineConfig):
        self.cfg = config
        self.lgb_params = {}
        self.xgb_params = {}
        self.cb_params = {}  # NEW

    def _cv3(self, model_fn, X, y) -> float:
        cv = StratifiedKFold(3, shuffle=True, random_state=self.cfg.random_state)
        aucs = []
        for tr, val in cv.split(X, y):
            m = model_fn()
            if isinstance(m, lgb.LGBMClassifier):
                m.fit(X[tr], y[tr], eval_set=[(X[val], y[val])],
                      callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)])
            elif isinstance(m, xgb.XGBClassifier):
                m.fit(X[tr], y[tr], eval_set=[(X[val], y[val])], verbose=False)
            else:  # CatBoost
                m.fit(X[tr], y[tr], eval_set=(X[val], y[val]), early_stopping_rounds=50)
            aucs.append(roc_auc_score(y[val], m.predict_proba(X[val])[:,1]))
        return float(np.mean(aucs))

    def _tune_lgb(self, X, y) -> Dict:
        print("Tuning LightGBM...")
        def obj(trial):
            p = {
                "objective":"binary","metric":"auc","verbosity":-1,"n_jobs":-1,"random_state":self.cfg.random_state,
                "n_estimators": trial.suggest_int("n_estimators",800,2000),
                "learning_rate": trial.suggest_float("learning_rate",0.005,0.05,log=True),
                "num_leaves": trial.suggest_int("num_leaves",20,150),
                "max_depth": trial.suggest_int("max_depth",4,12),
                "min_child_samples": trial.suggest_int("min_child_samples",20,100),
                "subsample": trial.suggest_float("subsample",0.6,1.0),
                "colsample_bytree": trial.suggest_float("colsample_bytree",0.6,1.0),
                "reg_alpha": trial.suggest_float("reg_alpha",1e-4,10,log=True),
                "reg_lambda": trial.suggest_float("reg_lambda",1e-4,10,log=True),
                "scale_pos_weight": trial.suggest_float("scale_pos_weight",1.0,5.0),
            }
            return self._cv3(lambda: lgb.LGBMClassifier(**p), X, y)
        s = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=self.cfg.random_state))
        s.optimize(obj, n_trials=self.cfg.optuna_trials, show_progress_bar=True)
        best = s.best_params
        best.update({"objective":"binary","metric":"auc","verbosity":-1,"n_jobs":-1,"random_state":self.cfg.random_state})
        print(f"LGB best AUC: {s.best_value:.4f}")
        return best

    def _tune_xgb(self, X, y) -> Dict:
        print("Tuning XGBoost...")
        spw = float((y==0).sum()/(y==1).sum())
        def obj(trial):
            p = {
                "objective":"binary:logistic","eval_metric":"auc","tree_method":"hist","n_jobs":-1,"verbosity":0,
                "early_stopping_rounds":50,"random_state":self.cfg.random_state,"scale_pos_weight": spw,
                "n_estimators": trial.suggest_int("n_estimators",800,2000),
                "learning_rate": trial.suggest_float("learning_rate",0.005,0.05,log=True),
                "max_depth": trial.suggest_int("max_depth",3,10),
                "min_child_weight": trial.suggest_int("min_child_weight",1,10),
                "subsample": trial.suggest_float("subsample",0.6,1.0),
                "colsample_bytree": trial.suggest_float("colsample_bytree",0.6,1.0),
                "gamma": trial.suggest_float("gamma",0.0,5.0),
                "reg_alpha": trial.suggest_float("reg_alpha",1e-4,10,log=True),
                "reg_lambda": trial.suggest_float("reg_lambda",1e-4,10,log=True),
            }
            return self._cv3(lambda: xgb.XGBClassifier(**p), X, y)
        s = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=self.cfg.random_state))
        s.optimize(obj, n_trials=self.cfg.optuna_trials, show_progress_bar=True)
        best = s.best_params
        best.update({"objective":"binary:logistic","eval_metric":"auc","tree_method":"hist","n_jobs":-1,"random_state":self.cfg.random_state,"scale_pos_weight":spw})
        print(f"XGB best AUC: {s.best_value:.4f}")
        return best

    # NEW: CatBoost tuning
    def _tune_catboost(self, X, y) -> Dict:
        print("Tuning CatBoost...")
        spw = float((y==0).sum()/(y==1).sum())
        def obj(trial):
            p = {
                "iterations": trial.suggest_int("iterations", 800, 2000),
                "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.05, log=True),
                "depth": trial.suggest_int("depth", 4, 10),
                "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-3, 10, log=True),
                "min_child_samples": trial.suggest_int("min_child_samples", 10, 100),
                "subsample": trial.suggest_float("subsample", 0.6, 1.0),
                "colsample_bylevel": trial.suggest_float("colsample_bylevel", 0.6, 1.0),
                "scale_pos_weight": spw, "random_seed": self.cfg.random_state, "verbose": False, "task_type": "CPU",
            }
            return self._cv3(lambda: CatBoostClassifier(**p), X, y)
        s = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=self.cfg.random_state))
        s.optimize(obj, n_trials=self.cfg.optuna_trials, show_progress_bar=True)
        best = s.best_params
        best.update({"verbose": False, "task_type": "CPU", "random_seed": self.cfg.random_state, "scale_pos_weight": spw})
        print(f"CatBoost best AUC: {s.best_value:.4f}")
        return best

    def run(self, X, y):
        if self.cfg.use_optuna:
            self.lgb_params = self._tune_lgb(X, y)
            self.xgb_params = self._tune_xgb(X, y)
            self.cb_params = self._tune_catboost(X, y)  # NEW
        else:
            self.lgb_params = dict(self.cfg.lgb_defaults)
            self.xgb_params = dict(self.cfg.xgb_defaults)
            self.cb_params = dict(self.cfg.cb_defaults)
            self.xgb_params["scale_pos_weight"] = float((y==0).sum()/(y==1).sum())
        return self.lgb_params, self.xgb_params, self.cb_params

tuner = HyperparamTuner(CFG)
lgb_params, xgb_params, cb_params = tuner.run(X, y)


Tuning LightGBM...


Best trial: 11. Best value: 0.783784: 100%|██████████| 30/30 [1:59:29<00:00, 238.99s/it]  


LGB best AUC: 0.7838
Tuning XGBoost...


Best trial: 29. Best value: 0.782171: 100%|██████████| 30/30 [3:20:28<00:00, 400.96s/it]  


XGB best AUC: 0.7822
Tuning CatBoost...


Best trial: 23. Best value: 0.781876: 100%|██████████| 30/30 [3:06:03<00:00, 372.11s/it]  

CatBoost best AUC: 0.7819


Step 7: EnsembleTrainer (ADD CatBoost + Stacking):


In [ ]:
from sklearn.linear_model import LogisticRegression

class EnsembleTrainer:
    def __init__(self, config: PipelineConfig):
        self.cfg = config
        self.final_lgb = None
        self.final_xgb = None
        self.final_cb = None
        self.final_meta = None
        self.metrics = {}

    def run(self, X, y, lgb_params, xgb_params, cb_params):
        kf = StratifiedKFold(self.cfg.cv_folds, shuffle=True, random_state=self.cfg.random_state)
        oof_lgb = np.zeros(len(y))
        oof_xgb = np.zeros(len(y))
        oof_cb = np.zeros(len(y))
        lgb_aucs, xgb_aucs, cb_aucs = [], [], []

        for fold, (tr, val) in enumerate(kf.split(X, y), 1):
            Xtr, Xval, ytr, yval = X[tr], X[val], y[tr], y[val]
            
            # LightGBM
            m_lgb = lgb.LGBMClassifier(**lgb_params)
            m_lgb.fit(Xtr, ytr, eval_set=[(Xval, yval)], callbacks=[lgb.early_stopping(50,verbose=False),lgb.log_evaluation(-1)])
            oof_lgb[val] = m_lgb.predict_proba(Xval)[:,1]
            lgb_aucs.append(roc_auc_score(yval, oof_lgb[val]))
            
            # XGBoost
            m_xgb = xgb.XGBClassifier(**xgb_params, early_stopping_rounds=50, verbosity=0)
            m_xgb.fit(Xtr, ytr, eval_set=[(Xval, yval)], verbose=False)
            oof_xgb[val] = m_xgb.predict_proba(Xval)[:,1]
            xgb_aucs.append(roc_auc_score(yval, oof_xgb[val]))
            
            # CatBoost
            m_cb = CatBoostClassifier(**cb_params)
            m_cb.fit(Xtr, ytr, eval_set=(Xval, yval), early_stopping_rounds=50)
            oof_cb[val] = m_cb.predict_proba(Xval)[:,1]
            cb_aucs.append(roc_auc_score(yval, oof_cb[val]))
            
            print(f"Fold {fold}: LGB={lgb_aucs[-1]:.4f} XGB={xgb_aucs[-1]:.4f} CB={cb_aucs[-1]:.4f}")

        # STACKING
        print("\nTraining stacking meta-learner...")
        oof_stack = np.column_stack([oof_lgb, oof_xgb, oof_cb])
        meta_model = LogisticRegression(C=1.0, random_state=self.cfg.random_state, max_iter=1000)
        meta_model.fit(oof_stack, y)
        oof_final = meta_model.predict_proba(oof_stack)[:,1]
        final_auc = roc_auc_score(y, oof_final)
        
        print(f"Stacking Ensemble OOF AUC: {final_auc:.4f}")
        print(f"Meta weights: LGB={meta_model.coef_[0][0]:.3f}, XGB={meta_model.coef_[0][1]:.3f}, CB={meta_model.coef_[0][2]:.3f}")

        # Final models trained on full data
        self.final_lgb = lgb.LGBMClassifier(**lgb_params)
        self.final_lgb.fit(X, y, callbacks=[lgb.log_evaluation(-1)])
        
        _xp = {k:v for k,v in xgb_params.items() if k != "early_stopping_rounds"}
        self.final_xgb = xgb.XGBClassifier(**_xp, verbosity=0)
        self.final_xgb.fit(X, y, verbose=False)
        
        _cbp = {k:v for k,v in cb_params.items() if "early_stopping" not in k}
        self.final_cb = CatBoostClassifier(**_cbp)
        self.final_cb.fit(X, y, verbose=False)
        
        self.final_meta = meta_model

        # ── NEW: Compute real metrics from OOF predictions ────────────────────
        from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                                     f1_score, brier_score_loss, average_precision_score,
                                     log_loss as sklearn_log_loss, roc_curve)
        from scipy.stats import ks_2samp

        # Best classification threshold via Youden's J (maximizes TPR - FPR)
        fpr_arr, tpr_arr, thr_arr = roc_curve(y, oof_final)
        best_thr = float(thr_arr[np.argmax(tpr_arr - fpr_arr)])
        oof_pred = (oof_final >= best_thr).astype(int)

        real_accuracy  = round(float(accuracy_score(y, oof_pred)), 4)
        real_precision = round(float(precision_score(y, oof_pred, zero_division=0)), 4)
        real_recall    = round(float(recall_score(y, oof_pred, zero_division=0)), 4)
        real_f1        = round(float(f1_score(y, oof_pred, zero_division=0)), 4)
        real_brier     = round(float(brier_score_loss(y, oof_final)), 4)
        real_logloss   = round(float(sklearn_log_loss(y, oof_final)), 4)
        real_auc_pr    = round(float(average_precision_score(y, oof_final)), 4)
        real_gini      = round(float(2 * final_auc - 1), 4)
        pos_rate       = round(float(np.mean(y)), 4)
        ks_val, _      = ks_2samp(oof_final[y == 1], oof_final[y == 0])
        real_ks        = round(float(ks_val), 4)

        print(f"\n── Real OOF Metrics (threshold={best_thr:.3f}) ──")
        print(f"  Accuracy:   {real_accuracy:.4f}")
        print(f"  Precision:  {real_precision:.4f}")
        print(f"  Recall:     {real_recall:.4f}")
        print(f"  F1 Score:   {real_f1:.4f}")
        print(f"  Brier:      {real_brier:.4f}")
        print(f"  KS Stat:    {real_ks:.4f}")
        print(f"  Gini:       {real_gini:.4f}")
        print(f"  AUC-PR:     {real_auc_pr:.4f}")
        print(f"  Pos. Rate:  {pos_rate:.4f}")
        # ── END NEW ───────────────────────────────────────────────────────────

        self.metrics = {
            "lgb_cv_auc_mean":  round(float(np.mean(lgb_aucs)), 4),
            "lgb_cv_auc_std":   round(float(np.std(lgb_aucs)),  4),
            "xgb_cv_auc_mean":  round(float(np.mean(xgb_aucs)), 4),
            "xgb_cv_auc_std":   round(float(np.std(xgb_aucs)),  4),
            "cb_cv_auc_mean":   round(float(np.mean(cb_aucs)),  4),
            "cb_cv_auc_std":    round(float(np.std(cb_aucs)),   4),
            "ensemble_oof_auc": round(float(final_auc), 4),
            "meta_weights":     {"lgb": float(meta_model.coef_[0][0]),
                                 "xgb": float(meta_model.coef_[0][1]),
                                 "cb":  float(meta_model.coef_[0][2])},
            # ── Real metrics (saved to ensemble_meta.json → read by dashboard) ──
            "accuracy":        real_accuracy,
            "precision":       real_precision,
            "recall":          real_recall,
            "f1":              real_f1,
            "brier_score":     real_brier,
            "log_loss":        real_logloss,
            "auc_pr":          real_auc_pr,
            "gini":            real_gini,
            "ks_stat":         real_ks,
            "positive_rate":   pos_rate,
            "best_threshold":  best_thr,
            "train_samples":   int(len(y)),
        }
        return self.final_lgb, self.final_xgb, self.final_cb, self.final_meta, self.metrics

trainer = EnsembleTrainer(CFG)
final_lgb, final_xgb, final_cb, meta_model, cv_metrics = trainer.run(X, y, lgb_params, xgb_params, cb_params)

Fold 1: LGB=0.7816 XGB=0.7810 CB=0.7801
Fold 2: LGB=0.7904 XGB=0.7886 CB=0.7887
Fold 3: LGB=0.7829 XGB=0.7822 CB=0.7815
Fold 4: LGB=0.7869 XGB=0.7861 CB=0.7869
Fold 5: LGB=0.7817 XGB=0.7794 CB=0.7786

Training stacking meta-learner...
Stacking Ensemble OOF AUC: 0.7849
Meta weights: LGB=1.800, XGB=1.785, CB=1.726



## 🔍 Step 8 — SHAPExplainer

In [13]:


import shap

class SHAPExplainer:
    def __init__(self, config: PipelineConfig):
        self.cfg = config
        self.importance = None

    def run(self, lgb_model, xgb_model, cb_model, X, feat_cols):
        n = min(self.cfg.shap_sample, len(X))
        idx = np.random.choice(len(X), n, replace=False)
        Xs = X[idx]

        print("Computing LGB SHAP...")
        lgb_expl = shap.TreeExplainer(lgb_model)
        lgb_shap = lgb_expl.shap_values(Xs)
        if isinstance(lgb_shap, list): lgb_shap = lgb_shap[1]

        print("Computing XGB SHAP...")
        xgb_expl = shap.TreeExplainer(xgb_model)
        xgb_shap = xgb_expl.shap_values(Xs)
        if isinstance(xgb_shap, list): xgb_shap = xgb_shap[1]
        
        # NEW: CatBoost SHAP
        print("Computing CatBoost SHAP...")
        cb_expl = shap.TreeExplainer(cb_model)
        cb_shap = cb_expl.shap_values(Xs)
        if isinstance(cb_shap, list): cb_shap = cb_shap[1]

        imp = pd.DataFrame({
            "feature": feat_cols,
            "lgb_shap_mean": np.abs(lgb_shap).mean(axis=0),
            "xgb_shap_mean": np.abs(xgb_shap).mean(axis=0),
            "cb_shap_mean": np.abs(cb_shap).mean(axis=0),  # NEW
        })
        imp["ensemble_shap"] = (imp["lgb_shap_mean"] + imp["xgb_shap_mean"] + imp["cb_shap_mean"]) / 3
        imp = imp.sort_values("ensemble_shap", ascending=False).reset_index(drop=True)

        print("\nTop 15 features:")
        for i, row in imp.head(15).iterrows():
            print(f"{i+1}. {row['feature']}: LGB={row['lgb_shap_mean']:.4f} XGB={row['xgb_shap_mean']:.4f} CB={row['cb_shap_mean']:.4f}")

        self.importance = imp
        return imp, lgb_shap, xgb_shap, cb_shap, Xs

explainer = SHAPExplainer(CFG)
shap_importance, lgb_shap, xgb_shap, cb_shap, X_sample = explainer.run(
    final_lgb, final_xgb, final_cb, X, feat_cols
)


Computing LGB SHAP...
Computing XGB SHAP...
Computing CatBoost SHAP...

Top 15 features:
1. EXT_SOURCE_MEAN: LGB=0.3920 XGB=0.4579 CB=0.3176
2. CREDIT_TERM: LGB=0.1343 XGB=0.1386 CB=0.1290
3. CODE_GENDER: LGB=0.1302 XGB=0.1324 CB=0.1379
4. POS_CNT_INSTALMENT_FUTURE_mean: LGB=0.1262 XGB=0.1260 CB=0.1070
5. GOODS_CREDIT_RATIO: LGB=0.1080 XGB=0.1085 CB=0.1220
6. POS_MONTHS_BALANCE_min: LGB=0.1010 XGB=0.1019 CB=0.0874
7. NAME_EDUCATION_TYPE: LGB=0.0910 XGB=0.0929 CB=0.0971
8. PAYMENT_RATIO_mean: LGB=0.0910 XGB=0.0871 CB=0.0848
9. PREV_CNT_PAYMENT_mean: LGB=0.0763 XGB=0.0828 CB=0.0819
10. AMT_ANNUITY: LGB=0.0704 XGB=0.0878 CB=0.0811
11. POS_MONTHS_BALANCE_mean: LGB=0.0794 XGB=0.0778 CB=0.0623
12. AMT_GOODS_PRICE: LGB=0.0648 XGB=0.0834 CB=0.0673
13. NAME_FAMILY_STATUS: LGB=0.0681 XGB=0.0681 CB=0.0650
14. EXT_SOURCE_2_3_PROD: LGB=0.0624 XGB=0.0609 CB=0.0662
15. FLAG_OWN_CAR: LGB=0.0646 XGB=0.0668 CB=0.0571


9: ModelSaver

In [16]:
class ModelSaver:
    def __init__(self, config: PipelineConfig):
        self.cfg = config

    def run(self, pipeline, lgb_model, xgb_model, cb_model, meta_model,
            feat_cols, lgb_params, xgb_params, cb_params, metrics, shap_importance, 
            lgb_shap, xgb_shap, cb_shap, X_sample) -> Dict:
        d = self.cfg.model_dir

        # Save original models
        joblib.dump(pipeline, d/"preprocessor.pkl")
        joblib.dump(lgb_model, d/"lgb_model.pkl")
        joblib.dump(xgb_model, d/"xgb_model.pkl")
        
        # NEW: Save CatBoost and meta model
        joblib.dump(cb_model, d/"cb_model.pkl")
        joblib.dump(meta_model, d/"meta_model.pkl")

        # Save SHAP values
        joblib.dump({
            "lgb_shap_values": lgb_shap.tolist(),
            "xgb_shap_values": xgb_shap.tolist(),
            "cb_shap_values": cb_shap.tolist(),  # NEW
            "X_sample": X_sample.tolist(),
            "feature_names": list(feat_cols),
            "importance_df": shap_importance.to_dict("records"),
        }, d/"shap_values.pkl")

        shap_importance.to_csv(d/"feature_importance.csv", index=False)

        def _clean(d):
            return {k: float(v) if isinstance(v,(float,np.floating)) else v for k,v in d.items()}

        meta = {
            "trained_at": datetime.now().isoformat(),
            "model_type": "LightGBM + XGBoost + CatBoost Stacking Ensemble",
            "n_features": len(feat_cols),
            "feature_columns": list(feat_cols),
            "metrics": metrics,
            "ensemble_auc": metrics["ensemble_oof_auc"],
            "lgb_params": _clean(lgb_params),
            "xgb_params": _clean(xgb_params),
            "cb_params": _clean(cb_params),  # NEW
            "meta_weights": metrics.get("meta_weights", {}),  # NEW
            "top_features": shap_importance.head(20)["feature"].tolist(),
        }
        with open(d/"ensemble_meta.json","w") as f:
            json.dump(meta, f, indent=2)

        print("Models saved!")
        return meta

saver = ModelSaver(CFG)
meta = saver.run(
    pipeline, final_lgb, final_xgb, final_cb, meta_model,
    feat_cols, lgb_params, xgb_params, cb_params,
    cv_metrics, shap_importance, lgb_shap, xgb_shap, cb_shap, X_sample
)


Models saved!


---
## 10 Pipeline Complete — Results

In [17]:
print("=" * 60)
print("  IMPROVED PIPELINE COMPLETE")
print("=" * 60)
print(f"Trained at: {meta['trained_at']}")
print(f"Ensemble OOF AUC: {meta['metrics']['ensemble_oof_auc']:.4f}")
print(f"LightGBM CV AUC: {meta['metrics']['lgb_cv_auc_mean']:.4f} ± {meta['metrics']['lgb_cv_auc_std']}")
print(f"XGBoost CV AUC: {meta['metrics']['xgb_cv_auc_mean']:.4f} ± {meta['metrics']['xgb_cv_auc_std']}")
print(f"CatBoost CV AUC: {meta['metrics']['cb_cv_auc_mean']:.4f} ± {meta['metrics']['cb_cv_auc_std']}")
print(f"Features: {meta['n_features']}")
print(f"Meta weights: {meta['meta_weights']}")
print()
print("Top 10 features:")
for i, f in enumerate(meta["top_features"][:10], 1):
    print(f"  {i:2d}. {f}")
print()
print("Models saved to: credit_scoring/models/")
print("=" * 60)


  IMPROVED PIPELINE COMPLETE
Trained at: 2026-03-07T06:03:52.797230
Ensemble OOF AUC: 0.7849
LightGBM CV AUC: 0.7847 ± 0.0035
XGBoost CV AUC: 0.7835 ± 0.0034
CatBoost CV AUC: 0.7831 ± 0.0039
Features: 190
Meta weights: {'lgb': 1.8003881548542326, 'xgb': 1.7849823117825272, 'cb': 1.7262852890815183}

Top 10 features:
   1. EXT_SOURCE_MEAN
   2. CREDIT_TERM
   3. CODE_GENDER
   4. POS_CNT_INSTALMENT_FUTURE_mean
   5. GOODS_CREDIT_RATIO
   6. POS_MONTHS_BALANCE_min
   7. NAME_EDUCATION_TYPE
   8. PAYMENT_RATIO_mean
   9. PREV_CNT_PAYMENT_mean
  10. AMT_ANNUITY

Models saved to: credit_scoring/models/
